# Fleet Delivery Fuel Consumption
Regression case study | 60 marks | 20 tests

Run the definition cells first. Run training and prediction separately from this folder. Prediction requires the saved model, but does not require training again.

In [3]:
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RAW_NUMERIC = ["distance_km", "trip_duration_hours", "payload_kg",
               "ambient_temperature_c", "idle_minutes"]
numeric_cols = RAW_NUMERIC
numeric_features = numeric_cols + ["start_hour", "average_speed_kmph"]
category_cols = ["depot_code", "vehicle_type", "route_type"]
features = numeric_features + category_cols
TARGET = "fuel_consumed_litres"

### load_data

In [4]:
def load_data(filename: str) -> pd.DataFrame:
    """Return a cleaned DataFrame with the original data and engineered features.

    Return type: pandas.DataFrame. The same function accepts train, test, or
    prediction data; it does not need a training flag.
    """
    df = pd.read_csv(filename)
    df = df.drop_duplicates(subset="trip_id", keep="first").copy()

    date=df["trip_start_ts"] 
    date =pd.to_datetime(
        date, format="%Y - %m - %d", errors="coerce"
    )
    remaining = date.isna()
    date.loc[remaining]=date =pd.to_datetime(
        date, format="mixed" ,dayfirst=True, errors="coerce"
    )
    df['trip_start_ts']=date
    df = df.dropna(subset=["trip_start_ts"]).copy()

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

    for col in category_cols:
        df[col] = (
            df[col].astype("string").str.strip().str.lower()
            .replace("", "unknown").fillna("unknown").astype(str)
        )

    df["start_hour"] = df["trip_start_ts"].dt.hour
    positive_duration = df["trip_duration_hours"].where(
        df["trip_duration_hours"] > 0
    )
    df["average_speed_kmph"] = (
        df["distance_km"] / positive_duration
    ).replace([np.inf, -np.inf], np.nan)

    return df.reset_index(drop=True)

### build_pipeline

In [5]:
def build_pipeline(algorithm) -> Pipeline:
    """Return an UNFITTED sklearn Pipeline containing preprocessing and model.

    Return type: sklearn.pipeline.Pipeline.
    """
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("category", OneHotEncoder(handle_unknown="ignore"), category_cols)
    ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", algorithm)
    ])

### prepare_datasets

In [6]:
def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Return (X_train, X_test, y_train, y_test).

    Return types, in order: DataFrame, DataFrame, Series, Series.
    Targets with missing, invalid or infinite values are removed.
    """
    train_df = load_data(train_file)
    test_df = load_data(test_file)

    for df in (train_df, test_df):
        df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
        df[TARGET] = df[TARGET].replace([np.inf, -np.inf], np.nan)
        df.dropna(subset=[TARGET], inplace=True)


    X_train, y_train = train_df[features], train_df[TARGET]
    X_test, y_test = test_df[features], test_df[TARGET]
    return X_train, X_test, y_train, y_test

### train_models

In [7]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    """Return dict[str, fitted sklearn.pipeline.Pipeline]."""
    models = {
        "linear_regression": LinearRegression(),
        "random_forest": RandomForestRegressor(
            n_estimators=150,
            max_depth=10,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    }

    trained_models = {}
    for name, algorithm in models.items():
        pipeline = build_pipeline(algorithm)
        pipeline.fit(X_train, y_train)
        trained_models[name] = pipeline

    return trained_models

### evaluate_models

In [8]:
def evaluate_models(trained_models: dict[str, Pipeline],
                    X_test: pd.DataFrame,
                    y_test: pd.Series) -> pd.DataFrame:
    """Return comparison DataFrame: model_name, mae, rmse, r2; RMSE ascending."""
    results = []
    for name, pipeline in trained_models.items():
        predictions = pipeline.predict(X_test)
        results.append({
            "model_name": name,
            "mae": mean_absolute_error(y_test, predictions),
            "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
            "r2": r2_score(y_test, predictions)
        })

    comparison = pd.DataFrame(results)
    comparison = comparison.sort_values("rmse").reset_index(drop=True)
    return comparison

### Function 6 — Select the best model

In [9]:
def select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    """Return the first-ranked model name and its existing fitted pipeline."""
    best_name = comparison.iloc[0]["model_name"]
    return best_name, trained_models[best_name]

### predict_fuel

In [10]:
def predict_fuel(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    """Return DataFrame[trip_id, predicted_fuel_consumed_litres].

    Clip negative estimates to zero, round to one decimal and preserve order.
    """
    predictions = model.predict(df[features])
    predictions = np.maximum(predictions, 0).round(1)
    return pd.DataFrame({
        "trip_id": df["trip_id"].to_numpy(),
        "predicted_fuel_consumed_litres": predictions
    })

### train

In [11]:
def train() -> tuple[str, pd.DataFrame]:
    """Fit, compare, persist; return (best_name: str, comparison: DataFrame)."""
    X_train, X_test, y_train, y_test = prepare_datasets(
        "fleet_train.csv", "fleet_test.csv"
    )
    trained_models = train_models(X_train, y_train)
    comparison = evaluate_models(trained_models, X_test, y_test)
    best_name, best_model = select_best_model(comparison, trained_models)

    joblib.dump(best_model, "best_model.pkl")
    comparison.to_csv("model_comparison.csv", index=False)
    return best_name, comparison

### predict

In [12]:
def predict() -> pd.DataFrame:
    """Load a fitted model and return the queued prediction DataFrame."""
    df = load_data("fleet_predict.csv")
    model = joblib.load("best_model.pkl")
    result = predict_fuel(model, df)
    result.to_csv("fleet_predictions.csv", index=False)

    return result

### Training workflow
Execute only when training or refreshing the model.

In [13]:
best_name, comparison = train()

/var/folders/bd/c55c6wf11p39nnf_s8wrsmzc0000gn/T/ipykernel_58239/537962942.py:13: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.dropna(subset=[TARGET], inplace=True)
/var/folders/bd/c55c6wf11p39nnf_s8wrsmzc0000gn/T/ipykernel_58239/537962942.py:13: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.dropna(subset=[TARGET], inplace=True)


ValueError: Found array with 0 sample(s) (shape=(0, 7)) while a minimum of 1 is required by SimpleImputer.

### Prediction workflow
Run the definitions and this cell in a fresh session after training has produced best_model.pkl.

In [ ]:
predictions = predict()